# BERTurk İnce Ayarı — 8 Sınıflı Kampanya Türü Sınıflandırması**Proje:** Anatolia AI — TEKNOFEST Türkçe Yapay Zekâ Dil Ajanları Yarışması, 2. Senaryo**Ortam:** Google Colab, ücretsiz katman (T4 GPU). Ücretli servis/API kullanılmaz.**Model:** `dbmdz/bert-base-turkish-cased` (BERTurk)---## Amaç`CLAUDE.md` §4, ince ayara (fine-tune) **yalnızca** 8 sınıflı kampanya türüsınıflandırması için izin verir; alan çıkarımı (NER) kural + few-shot LLM ileyapılır. Bu defter o tek izinli ince ayarı uçtan uca koşar ve sonucu, kuraltabanlı temel çizginin **ölçülmüş** sayısıyla yan yana koyar.**Geçilmesi gereken sayı (temel çizgi, ölçülmüş):**| Kol | accuracy | makro-F1 | çekimser | n ||---|---|---|---|---|| `RuleHintClassifier` (kural) | **0,700** | **0,762** | 1 | 20 |Bu sayı uydurulmadı; depoda `python -m scripts.eval_classifier` komutunun`data/gold/gold.v1.json` üzerinde ürettiği çıktıdır. Defterin son bölümüBERTurk'ü **aynı gold küme, aynı metrik, aynı çekimser muamelesi** ile ölçer —başka türlüsü kıyaslanabilir sayı üretmez.---## Lisans doğrulaması (yarışma kuralı gereği kritik)`CLAUDE.md` §3 ve §20: proje **yalnızca Apache-2.0 veya MIT** lisanslı modelağırlığı kullanabilir. **Gemma** ve **Llama community** lisanslı ağırlıklarkullanım kısıtı içerdiği için yasaktır (diskalifiye riski).Bu defter yazılmadan önce aday model bağımsız olarak doğrulandı:| Alan | Değer | Kaynak ||---|---|---|| Model | `dbmdz/bert-base-turkish-cased` | — || `cardData.license` | **`mit`** | Hugging Face model API'si || `base_model` | **beyan edilmemiş** → model zincirin **kökü** | Hugging Face model API'si || Karar | ✅ **UYGUN** (MIT, izinli listede) | — |**Kanıt URL'leri:**- Model kartı: <https://huggingface.co/dbmdz/bert-base-turkish-cased>- Makine okunur metaveri: <https://huggingface.co/api/models/dbmdz/bert-base-turkish-cased>**`base_model` zinciri:** Model kartında `base_model` alanı **yoktur**. BERTurksıfırdan eğitilmiş bir Türkçe BERT'tir, başka bir modelin türevi değildir.Dolayısıyla takip edilecek bir zincir yoktur ve "türev modelin lisansı kökündendaha serbest olamaz" tuzağı burada oluşmaz. Bu, deponun mevcut`docs/model-license-audit.md` kaydıyla da tutarlıdır (BERTurk satırı: MIT,taban: "kök").> **Neden bu ayrım önemli:** `docs/model-license-audit.md`, NuExtract-2.0-4B> örneğinde `license` etiketi temiz görünen bir modelin taban zincirinin kirli> çıkabildiğini kayda geçirmiş. Bu yüzden etiket okumak yetmez, zincir sorulur.> BERTurk'te zincir yok — bu bir varsayım değil, aşağıdaki hücrenin canlı> doğruladığı bir olgu.**Bu defter yarışma kurallarına neden uygun:**1. Ağırlık lisansı **MIT** — izinli listede, kullanım kısıtı yok.2. Ücretli servis/API yok; yalnızca Colab ücretsiz katmanı.3. Eğitim çevrimiçi yapılır, ama **çıkarım yerelde ve çevrimdışı** koşar —   son bölümdeki `local_files_only=True` hücresi bunu kanıtlar.4. Üretilen ağırlıklar projenin Apache-2.0 dağıtımına engel değildir; MIT,   Apache-2.0 ile uyumludur (atıf yükümlülüğü korunur).

### Lisans kapısı — çalıştırılabilir doğrulamaYukarıdaki tablo elle yazılmış bir iddiadır. Aşağıdaki hücre onu **canlı**doğrular ve lisans izinli listede değilse defteri `RuntimeError` ile durdurur.Böylece model kartı ileride değişirse defter sessizce yanlış bir modelieğitmez.

In [ ]:
# --- LİSANS KAPISI: geçemezse defter burada durur -------------------------import json, urllib.requestMODEL_ADI = "dbmdz/bert-base-turkish-cased"IZINLI_LISANSLAR = {"mit", "apache-2.0"}          # CLAUDE.md §3, §20def lisans_dogrula(model_adi: str, derinlik: int = 0) -> str:    """Modelin lisansını doğrular ve base_model zincirini KÖKE kadar takip eder.    Türev bir modelin lisansı kökündekinden daha serbest olamaz; bu yüzden    zincirdeki HER halka izinli listede olmak zorundadır.    """    girinti = "  " * derinlik    url = f"https://huggingface.co/api/models/{model_adi}"    with urllib.request.urlopen(url, timeout=30) as r:        meta = json.load(r)    kart = meta.get("cardData") or {}    lisans = (kart.get("license") or "").strip().lower()    taban = kart.get("base_model")    print(f"{girinti}model      : {model_adi}")    print(f"{girinti}license    : {lisans or '(BEYAN EDİLMEMİŞ)'}")    print(f"{girinti}base_model : {taban or '(yok → zincirin KÖKÜ)'}")    if lisans not in IZINLI_LISANSLAR:        raise RuntimeError(            f"LİSANS UYGUN DEĞİL: {model_adi} → '{lisans}'. "            f"İzinli: {sorted(IZINLI_LISANSLAR)}. CLAUDE.md §3/§20 gereği "            f"bu ağırlık kullanılamaz; defteri koşma."        )    # Zinciri köke kadar takip et (str veya liste olabilir).    if taban:        for t in ([taban] if isinstance(taban, str) else taban):            if t and t != model_adi:                print(f"{girinti}↳ taban zinciri takip ediliyor...")                lisans_dogrula(t, derinlik + 1)    return lisans_lisans = lisans_dogrula(MODEL_ADI)print()print(f"✅ LİSANS UYGUN: {MODEL_ADI} → {_lisans.upper()} (zincirin tamamı temiz)")print("   Kanıt: https://huggingface.co/dbmdz/bert-base-turkish-cased")

---## 1. Ortam: GPU kontrolü ve bağımlılıklarColab'da **Çalışma zamanı → Çalışma zamanı türünü değiştir → Donanımhızlandırıcı: T4 GPU** seçili olmalı. GPU yoksa defter yine koşar ama eğitimCPU'da onlarca dakika sürer.

In [ ]:
# --- GPU kontrolü ---------------------------------------------------------import subprocess, systry:    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",                          "--format=csv"], capture_output=True, text=True,                         timeout=60).stdout.strip())except FileNotFoundError:    print("nvidia-smi yok → GPU atanmamış olabilir.")print(f"\nPython: {sys.version.split()[0]}")

In [ ]:
# --- Bağımlılıklar (sürümler PİNLİ: defter aylar sonra da aynı davransın) --# Not: Colab'ın kendi torch'una dokunulmuyor; üstüne kurmak çalışma zamanını# bozar ve ücretsiz katmanda yeniden başlatma gerektirir.!pip install -q \    "transformers==4.44.2" \    "datasets==2.21.0" \    "accelerate==0.34.2" \    "scikit-learn==1.5.2" \    "numpy<2.0"import torch, transformers, datasets, sklearn, numpy as npprint("torch        :", torch.__version__)print("transformers :", transformers.__version__)print("datasets     :", datasets.__version__)print("scikit-learn :", sklearn.__version__)print("numpy        :", np.__version__)print("CUDA         :", torch.cuda.is_available(),      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

---## 2. Veri: gümüş küme (n=505, 8 sınıf)### Hangi dosya yüklenecekColab'a **tek bir dosya** yüklenir:```berturk_egitim.jsonl      # 505 satır: {doc_id, text, label, bank_slug}berturk_gold_eval.jsonl   #  20 satır: {doc_id, text, label}  ← temel çizgi karşılaştırması için```Bu iki dosya depoda **hazır durmuyor**; aşağıdaki *yerel hazırlık* hücresionları üretir. Sebebi: etiketler ile metinler depoda **ayrı dosyalarda**tutuluyor ve birleştirilmeleri gerekiyor:| Dosya | Ne var | Git'te? ||---|---|---|| `data/silver/silver.jsonl` | 505 kayıt: `doc_id`, `label`, `confidence`, `evidence`, `votes` — **metin yok** | ✅ izleniyor || `data/silver/trainable.jsonl` | 660 kayıt: `doc_id`, `text`, `core_text` — 505'in **491**'ini kapsar | ❌ `.gitignore` s.76 || `data/raw-classic/<banka>/{live,products}/<slug>.txt` | ham metin | ✅ izleniyor (yalnız `.html` yok sayılır) |Ölçülmüş çözünürlük:| Kaynak | Kapsam | Metin kalitesi ||---|---|---|| `trainable.jsonl` → `core_text` | 505'in **491**'i | **iyi** — menü/altbilgi kalıpları ayıklanmış || `data/raw-classic/.../*.txt` | 505'in **505**'i (`live` 416 + `products` 89) | ham — kalıp metin içeriyor |**505/505 çözülüyor, hiçbir kayıt düşmüyor.** Hazırlık hücresi önce`core_text`'i dener, bulamazsa ham `.txt`'e düşer.> **Taze klon uyarısı:** `trainable.jsonl` depoda **izlenmiyor** (12 MB, boyut> nedeniyle yok sayılmış). Yeni klonlanmış bir kopyada hücre 505 belgenin> tamamını ham `.txt`'ten çözer — çalışır, ama metinler kalıp içerdiği için> sonuç `core_text` ile birebir aynı olmaz. Hücre hangi modda koştuğunu basar.> `core_text` kapsamı dışında kalan **14** belgenin hepsi kredi *ürün* sayfası> (taşıt/konut/kentsel dönüşüm) ve hepsi `denizbank`, `ing`, `teb` bankalarına> ait. Atlanırlarsa `Taşıt Finansmanı` ve `Konut Finansmanı` sınıfları> orantısız zarar görür — bu yüzden geri kazanılıyorlar.### `core_text` mi `text` mi`core_text`, kaynak sayfadan menü/altbilgi kalıplarının (boilerplate)ayıklanmış hâlidir. Ölçülmüş uzunluklar (505 kayıt):| Alan | medyan | ortalama | maks ||---|---|---|---|| `text` | 5 043 krk | 12 543 krk | 102 557 krk || `core_text` | 1 718 krk | 3 948 krk | 90 280 krk |BERT'in 512 alt-sözcük (subword) penceresine `text` sığmaz; `core_text`sığmaya çok daha yakındır ve sinyal yoğunluğu yüksektir. **`core_text`kullanılır**, ham `.txt`'ten çözülen 14 belge için ayıklanmış sürümbulunmadığından ham metin baştan kırpılır.

### 2a. YEREL HAZIRLIK — bu hücreyi **depoda** koş, Colab'da değilBu hücreyi depo kökünde (`app/`) çalıştır — Jupyter'de aç ve koş, ya daiçeriğini bir dosyaya kopyalayıp `.venv/bin/python <dosya>` de. Colab'dakoşulursa ilk `assert` ile durur (depo orada yoktur).Çıktı iki dosya: `data/eval/berturk_egitim.jsonl` (505 satır) ve`data/eval/berturk_gold_eval.jsonl` (20 satır). Bu ikisini Colab'a yükle.> Bu iki dosya depoda **commit'li** durur; hücreyi koşmadan da doğrudan> Colab'a yükleyebilirsin. Hücre, veri değiştiğinde onları yeniden üretmek ve> sızıntı kontrolünü tekrarlamak için vardır.

In [ ]:
# --- YEREL HAZIRLIK: depo kökünde (app/) koş, Colab'da DEĞİL ---------------# Etiket + metin birleştirme. 505/505 kayıt çözülür (ölçülmüştür).import json, os, sys, collectionsDEPO = os.getcwd()          # app/ dizini olmalıassert os.path.exists(os.path.join(DEPO, "data", "silver", "silver.jsonl")), \    "Bu hücre depo kökünde (app/) koşmalı."sys.path.insert(0, DEPO)CIKTI_DIR = os.path.join(DEPO, "data", "eval")os.makedirs(CIKTI_DIR, exist_ok=True)# 1) Metin havuzu: trainable.jsonl (İZLENMİYOR — taze klonda olmayabilir)metin = {}yol_tr = os.path.join(DEPO, "data/silver/trainable.jsonl")if os.path.exists(yol_tr):    with open(yol_tr, encoding="utf-8") as fh:        for satir in fh:            d = json.loads(satir)            if d.get("core_text"):                metin[d["doc_id"]] = d["core_text"]    print(f"trainable.jsonl bulundu — {len(metin)} belge için ayıklanmış core_text var.")else:    print("trainable.jsonl YOK (gitignore s.76). Tüm metinler ham .txt'ten "          "çözülecek — çalışır, ama kalıp metin ayıklanmamış olur.")# 2) Gümüş etiketler + eksik metinler için raw-classic geri düşüşükaynak_sayaci = collections.Counter()egitim, cozulemeyen = [], []with open(os.path.join(DEPO, "data/silver/silver.jsonl"), encoding="utf-8") as fh:    for satir in fh:        r = json.loads(satir)        did = r["doc_id"]        banka, _, slug = did.partition("--")        t = metin.get(did)        if t:            kaynak_sayaci["trainable.core_text"] += 1        else:            for alt in ("live", "products", "manual"):                p = os.path.join(DEPO, "data/raw-classic", banka, alt, f"{slug}.txt")                if os.path.exists(p):                    with open(p, encoding="utf-8") as g:                        t = g.read()                    kaynak_sayaci[f"raw-classic/{alt}"] += 1                    break        if not t:            cozulemeyen.append(did)            continue        egitim.append({"doc_id": did, "text": t, "label": r["label"],                       "bank_slug": banka, "confidence": r["confidence"]})print(f"çözülen : {len(egitim)} / 505")print(f"kaynak  : {dict(kaynak_sayaci)}")if cozulemeyen:    print(f"UYARI — çözülemeyen {len(cozulemeyen)}: {cozulemeyen[:5]}")yol_egitim = os.path.join(CIKTI_DIR, "berturk_egitim.jsonl")with open(yol_egitim, "w", encoding="utf-8") as fh:    for r in egitim:        fh.write(json.dumps(r, ensure_ascii=False) + "\n")print(f"-> {yol_egitim}")# 3) Gold değerlendirme kümesi (temel çizgiyle AYNI küme)from scripts.gold_schema import load_goldgold = [r for r in load_gold(os.path.join(DEPO, "data/gold/gold.v1.json"))        if isinstance(r.campaign_type, str) and r.campaign_type]yol_gold = os.path.join(CIKTI_DIR, "berturk_gold_eval.jsonl")with open(yol_gold, "w", encoding="utf-8") as fh:    for r in gold:        fh.write(json.dumps({"doc_id": r.id, "text": r.text,                             "label": r.campaign_type}, ensure_ascii=False) + "\n")print(f"-> {yol_gold}  ({len(gold)} belge)")# 4) SIZINTI KONTROLÜ — gold belgeleri eğitim kümesine karışmamalıkesisim = {r["doc_id"] for r in egitim} & {r.id for r in gold}print(f"\nsızıntı (gold ∩ eğitim): {len(kesisim)} {sorted(kesisim)[:5]}")assert not kesisim, "SIZINTI! Gold belgesi eğitim kümesinde — ölçüm geçersiz."print("✅ Sızıntı yok.")

### 2b. Colab'da veri yükleme — iki yol**(a) Google Drive'a elle yüklenen dosya (önerilen).** İki `.jsonl` dosyasınıDrive'da `MyDrive/anatolia-ai/` klasörüne kopyala, sonra aşağıdaki hücrede`YOL = "drive"` bırak.**(b) Depodan doğrudan.** Depo yereldeyse `files.upload()` ile tarayıcıdanyükle (`YOL = "yukle"`). Depo herkese açık bir git uzağındaysa `git clone` dayapılabilir — ama bu depo özel olduğu için varsayılan yol Drive'dır.

In [ ]:
# --- Colab veri yükleme ---------------------------------------------------import json, osYOL = "drive"      # "drive" | "yukle"DRIVE_KLASOR = "/content/drive/MyDrive/anatolia-ai"if YOL == "drive":    from google.colab import drive    drive.mount("/content/drive")    yol_egitim = f"{DRIVE_KLASOR}/berturk_egitim.jsonl"    yol_gold   = f"{DRIVE_KLASOR}/berturk_gold_eval.jsonl"elif YOL == "yukle":    from google.colab import files    print("İki dosyayı birlikte seç: berturk_egitim.jsonl + berturk_gold_eval.jsonl")    files.upload()    yol_egitim, yol_gold = "berturk_egitim.jsonl", "berturk_gold_eval.jsonl"else:    raise ValueError("YOL 'drive' veya 'yukle' olmalı.")def jsonl_oku(p):    with open(p, encoding="utf-8") as fh:        return [json.loads(s) for s in fh if s.strip()]kayitlar = jsonl_oku(yol_egitim)gold_kayitlar = jsonl_oku(yol_gold)print(f"eğitim kümesi : {len(kayitlar)} kayıt")print(f"gold kümesi   : {len(gold_kayitlar)} kayıt")assert len(kayitlar) == 505, f"BEKLENEN 505, GELEN {len(kayitlar)} — hazırlık hücresini kontrol et."assert len(gold_kayitlar) == 20, f"BEKLENEN 20, GELEN {len(gold_kayitlar)}."print("\nörnek kayıt:", {k: (v[:80] + "..." if k == "text" else v)                         for k, v in kayitlar[0].items()})

### 2c. Sınıf dağılımı — bölmeden **ÖNCE**

In [ ]:
# --- Sınıf dağılımı: BÖLMEDEN ÖNCE ----------------------------------------import collections# Depodaki kanonik sıra (src/schemas.py CAMPAIGN_TYPES) — etiket kimliği# defterle depo arasında birebir aynı olmak zorunda.CAMPAIGN_TYPES = ["Finansman", "İhtiyaç Finansmanı", "Konut Finansmanı",                  "Taşıt Finansmanı", "Kart", "Alışveriş Puanı",                  "Yeni Müşteri", "Yatırım Ürünü"]etiketler = [r["label"] for r in kayitlar]bilinmeyen = set(etiketler) - set(CAMPAIGN_TYPES)assert not bilinmeyen, f"Şemada olmayan etiket: {bilinmeyen}"dagilim = collections.Counter(etiketler)toplam = len(etiketler)print(f"BÖLMEDEN ÖNCE — n={toplam}, {len(dagilim)} sınıf\n")print(f"{'sınıf':<22}{'adet':>6}{'oran':>9}")for s in CAMPAIGN_TYPES:    print(f"{s:<22}{dagilim[s]:>6}{dagilim[s]/toplam:>8.1%}")print(f"\nen küçük sınıf: {min(dagilim.values())}  |  en büyük: {max(dagilim.values())}")print(f"dengesizlik oranı: {max(dagilim.values())/min(dagilim.values()):.2f}x")print("\nbanka dağılımı:", dict(collections.Counter(r["bank_slug"] for r in kayitlar)))print("güven dağılımı :", dict(collections.Counter(r["confidence"] for r in kayitlar)))

### 2d. Bölme (split) — katmanlı, sabit tohum**Oranlar:** %70 eğitim / %15 doğrulama / %15 test.**`random_state = 42`** — sabit, defter her koşuşta aynı bölmeyi üretir.**Neden katmanlı (stratified):** en küçük sınıf 41 örnekli. Rastgele bölmede%15'lik test kümesine o sınıftan 2-3 örnek düşebilir, hatta hiç düşmeyebilir.Katmanlı bölme her sınıfın oranını üç parçada da korur; makro-F1 sınıf başınaF1'lerin ortalaması olduğu için bir sınıfın test kümesinde temsil edilmemesimetriği anlamsızlaştırır.**Uyarı — test kümesi küçük:** %15 × 505 ≈ 76 belge, sınıf başına ortalama 9,5örnek. Bu bir *sıralama sinyali* üretir, kesin performans ölçüsü değil. Bölüm6'daki güven aralığı hücresi bunu sayısallaştırır.

In [ ]:
# --- Katmanlı bölme -------------------------------------------------------from sklearn.model_selection import train_test_splitTOHUM = 42          # SABİT — tekrarlanabilirlik için asla değiştirmeX = list(range(len(kayitlar)))y = [r["label"] for r in kayitlar]# Önce test'i ayır (%15), kalandan doğrulamayı ayır (%15/%85 ≈ 0,1765)idx_kalan, idx_test = train_test_split(    X, test_size=0.15, random_state=TOHUM, stratify=y)idx_egitim, idx_dogrulama = train_test_split(    idx_kalan, test_size=0.15 / 0.85, random_state=TOHUM,    stratify=[y[i] for i in idx_kalan])bolme = {"eğitim": idx_egitim, "doğrulama": idx_dogrulama, "test": idx_test}# Bölmeler ayrık mı?assert not (set(idx_egitim) & set(idx_dogrulama)), "eğitim/doğrulama örtüşüyor"assert not (set(idx_egitim) & set(idx_test)),      "eğitim/test örtüşüyor"assert not (set(idx_dogrulama) & set(idx_test)),   "doğrulama/test örtüşüyor"assert len(idx_egitim) + len(idx_dogrulama) + len(idx_test) == len(kayitlar)print("BÖLMEDEN SONRA\n")baslik = f"{'sınıf':<22}" + "".join(f"{ad:>12}" for ad in bolme) + f"{'toplam':>9}"print(baslik); print("-" * len(baslik))for s in CAMPAIGN_TYPES:    satir, tpl = f"{s:<22}", 0    for ad, idx in bolme.items():        n = sum(1 for i in idx if y[i] == s)        tpl += n        satir += f"{n:>6}({n/len(idx):>5.1%})"    print(satir + f"{tpl:>9}")print("-" * len(baslik))print(f"{'TOPLAM':<22}" + "".join(f"{len(i):>12}" for i in bolme.values())      + f"{len(kayitlar):>9}")# Her sınıf her bölmede temsil ediliyor mu?for ad, idx in bolme.items():    eksik = set(CAMPAIGN_TYPES) - {y[i] for i in idx}    if eksik:        print(f"\n⚠️  '{ad}' bölmesinde TEMSİL EDİLMEYEN sınıf: {eksik}")    else:        print(f"✅ '{ad}': 8 sınıfın hepsi temsil ediliyor.")

---## 3. Hiperparametreler — açık ve gerekçeli| Parametre | Değer | Gerekçe ||---|---|---|| `learning_rate` | `2e-5` | BERT ince ayarında olağan aralık 2e-5 – 5e-5. **Alt uç** seçildi: 353 eğitim örneğiyle yüksek öğrenme oranı ilk epoch'ta önceden eğitilmiş temsilleri bozar (catastrophic forgetting). || `num_train_epochs` | `8` (erken durdurmayla) | Küçük kümede 2-3 epoch yetersiz kalabilir; 8 üst sınırdır. Gerçek durma noktasını **doğrulama makro-F1'i** belirler (`EarlyStoppingCallback`, sabır=3). Sabit epoch sayısı ya az öğrenir ya ezberler. || `per_device_train_batch_size` | `16` | 353 örnek → epoch başına ~22 adım. Daha büyük parti adım sayısını tek haneye düşürür, öğrenme oranı zamanlaması işlemez. T4 belleğine 16×256 rahat sığar. || `max_length` | `256` | Aşağıdaki hücre gerçek token uzunluklarını **ölçer**; 256'nın kapsama oranı orada basılır. BERT'in mimari üst sınırı 512'dir, ama 512 hem 2× yavaştır hem de kuyruktaki kalıp metni modele geri sokar. || `weight_decay` | `0.01` | Standart L2 düzenlileştirme; küçük kümede aşırı öğrenmeye karşı ucuz sigorta. || `warmup_ratio` | `0.1` | İlk %10 adımda öğrenme oranı kademeli açılır; küçük kümede ilk adımların şoku kalıcı hasar bırakır. || `SINIF_AGIRLIGI` | `True` | Kart (144) / Finansman (41) arası **3,51×** dengesizlik var. Metrik makro-F1 olduğu için küçük sınıflar büyükler kadar önemli; ağırlıksız kayıp modeli çoğunluk sınıfına iter. |### Aşırı öğrenme (overfitting) riski — açıkça**505 kayıt küçük bir kümedir.** 110 milyon parametreli bir modeli 353 örnekleince ayarlamak, modelin eğitim kümesini ezberlemesi için fazlasıyla yeterlikapasite bırakır. Alınan önlemler:1. **Erken durdurma**, doğrulama makro-F1'i üzerinden — eğitim kaybı düşerken   doğrulama metriği düzleşirse eğitim kesilir.2. **`load_best_model_at_end=True`** — son epoch'un değil, doğrulamada en iyi   epoch'un ağırlıkları alınır.3. **Ayrı test kümesi** — doğrulama kümesi erken durdurma kararında kullanıldığı   için üzerinde ölçülen sayı iyimserdir; nihai sayı hiç görülmemiş test   kümesinden okunur.4. **Bağımsız gold küme** — bölüm 5, eğitimle hiç teması olmayan `gold.v1.json`   üzerinde ölçer.Bu önlemlerin hiçbiri "küçük küme" sorununu **çözmez**, sadece görünür kılar.Bölüm 6'daki uyarı hücresi bunu açıkça söyler.

In [ ]:
# --- Tokenizer + max_length'in ÖLÇÜLMÜŞ gerekçesi -------------------------import numpy as npfrom transformers import AutoTokenizerMODEL_ADI = "dbmdz/bert-base-turkish-cased"tokenizer = AutoTokenizer.from_pretrained(MODEL_ADI)uzunluklar = np.array([len(tokenizer(r["text"], truncation=False,                                     add_special_tokens=True)["input_ids"])                       for r in kayitlar])print(f"Token uzunlukları (n={len(uzunluklar)}):")print(f"  medyan : {np.median(uzunluklar):>8.0f}")print(f"  ortalama: {uzunluklar.mean():>8.0f}")print(f"  maks   : {uzunluklar.max():>8.0f}")print()print(f"{'max_length':>12}{'tam sığan belge':>18}{'oran':>9}")for m in (128, 256, 384, 512):    n = int((uzunluklar <= m).sum())    print(f"{m:>12}{n:>18}{n/len(uzunluklar):>8.1%}")MAX_LENGTH = 256print(f"\nSEÇİLEN max_length = {MAX_LENGTH}")print(f"  belgelerin %{(uzunluklar <= MAX_LENGTH).mean()*100:.1f}'i tam sığıyor;")print(f"  kalanı BAŞTAN {MAX_LENGTH} token alınır (kampanya sayfalarında tür")print( "  sinyali başlıkta ve ilk paragrafta yoğunlaşır).")

---## 4. Metrikler — depo ile **birebir** aynıKarşılaştırmanın anlamlı olması için defterin metriği `scripts/eval_classifier.py`ile aynı olmak zorunda. Depodaki tanımlar (`eval/run_eval.py`) aynen burayataşınır:1. **Makro-F1**, `support > 0` olan sınıflar üzerinden ortalanır. Gold'da hiç   örneği olmayan sınıfın recall'ı tanımsızdır; onu 0 sayıp ortalamaya katmak   makro-F1'i gold kümesinin kapsamına göre keyfî biçimde düşürür.2. **Çekimserlik (abstain)** doğru sınıf için **FN** sayılır, hiçbir sınıf için   **FP** sayılmaz. Böylece susmak recall'u düşürür ama precision'ı şişirmez —   zor belgelerde susup kolaylarda konuşan bir model ödüllendirilmez.3. **Accuracy** = doğru / toplam; çekimser cevaplar yanlış sayılır.> BERTurk softmax'ı her zaman bir sınıf seçer, yani doğal olarak çekimser> kalmaz. `CEKIMSER_ESIGI` ile bir güven eşiği koyulabilir (0,0 = çekimserlik> yok). Eşik yükseltmek makro-F1'i **düşürür** çünkü çekimserlik FN'dir —> metrik, susarak yüksek skor almayı imkânsız kılacak biçimde tasarlanmıştır.

In [ ]:
# --- Metrikler: eval/run_eval.py ile birebir aynı --------------------------from dataclasses import dataclassfrom collections import Counterfrom typing import Optional@dataclassclass Counts:    tp: int = 0    fp: int = 0    fn: int = 0    def precision(self):        d = self.tp + self.fp        return self.tp / d if d else 0.0    def recall(self):        d = self.tp + self.fn        return self.tp / d if d else 0.0    def f1(self):        p, r = self.precision(), self.recall()        return 2 * p * r / (p + r) if (p + r) else 0.0    @property    def support(self):        return self.tp + self.fndef macro_f1(tablo: dict) -> float:    """Yalnız gold desteği (support > 0) olan sınıflar üzerinden ortalama."""    skor = [c.f1() for c in tablo.values() if c.support > 0]    return sum(skor) / len(skor) if skor else 0.0def olc(gercekler, tahminler):    """scripts/eval_classifier.py:olc ile aynı mantık.    tahmin None ise ÇEKİMSER: doğru sınıf için FN, hiçbir sınıf için FP değil.    """    tablo = {s: Counts() for s in CAMPAIGN_TYPES}    dogru = cekimser = 0    karisiklik = Counter()    for gercek, pred in zip(gercekler, tahminler):        if pred is None:            cekimser += 1            tablo[gercek].fn += 1            karisiklik[(gercek, "(çekimser)")] += 1            continue        if pred == gercek:            dogru += 1            tablo[gercek].tp += 1        else:            tablo[gercek].fn += 1            if pred in tablo:                tablo[pred].fp += 1            karisiklik[(gercek, pred)] += 1    n = len(gercekler)    return {"n": n, "dogru": dogru, "cekimser": cekimser,            "accuracy": dogru / n if n else 0.0,            "macro_f1": macro_f1(tablo),            "tablo": tablo, "karisiklik": karisiklik}def yazdir(ad, s):    print(f"\n=== {ad} ===")    print(f"n={s['n']}  doğru={s['dogru']}  çekimser={s['cekimser']}")    print(f"accuracy = {s['accuracy']:.3f}")    print(f"macro-F1 = {s['macro_f1']:.3f}")    print(f"\n{'sınıf':<22}{'destek':>7}{'P':>8}{'R':>8}{'F1':>8}")    for sinif in CAMPAIGN_TYPES:        c = s["tablo"][sinif]        if c.support == 0 and c.fp == 0:            continue        print(f"{sinif:<22}{c.support:>7}{c.precision():>8.3f}"              f"{c.recall():>8.3f}{c.f1():>8.3f}")    if s["karisiklik"]:        print("\nkarışıklıklar (gerçek -> tahmin):")        for (g, p), n in s["karisiklik"].most_common(10):            print(f"  {g} -> {p}: {n}")print("✅ Metrik fonksiyonları hazır (depo ile birebir).")

In [ ]:
# --- Veri kümesi hazırlama ------------------------------------------------import numpy as npfrom datasets import Datasetetiket2id = {s: i for i, s in enumerate(CAMPAIGN_TYPES)}id2etiket = {i: s for s, i in etiket2id.items()}def kume(idx):    return Dataset.from_dict({        "text":  [kayitlar[i]["text"]  for i in idx],        "label": [etiket2id[kayitlar[i]["label"]] for i in idx],        "doc_id": [kayitlar[i]["doc_id"] for i in idx],    })def tokenle(ornek):    return tokenizer(ornek["text"], truncation=True,                     max_length=MAX_LENGTH, padding=False)ds = {ad: kume(idx).map(tokenle, batched=True, remove_columns=["text"])      for ad, idx in bolme.items()}for ad, d in ds.items():    print(f"{ad:<12} {len(d):>4} örnek")

In [ ]:
# --- Model + sınıf ağırlıklı Trainer --------------------------------------import torch, numpy as npfrom torch import nnfrom transformers import (AutoModelForSequenceClassification, TrainingArguments,                          Trainer, DataCollatorWithPadding, EarlyStoppingCallback,                          set_seed)set_seed(TOHUM)SINIF_AGIRLIGI = True     # dengesizlik 3,51× — makro-F1 için açıkmodel = AutoModelForSequenceClassification.from_pretrained(    MODEL_ADI, num_labels=len(CAMPAIGN_TYPES),    id2label=id2etiket, label2id=etiket2id)# Ağırlık = n / (sınıf_sayısı * sınıf_adedi)  — sklearn 'balanced' formülüegitim_etiketleri = np.array(ds["eğitim"]["label"])adet = np.bincount(egitim_etiketleri, minlength=len(CAMPAIGN_TYPES))agirlik = len(egitim_etiketleri) / (len(CAMPAIGN_TYPES) * np.maximum(adet, 1))print("sınıf ağırlıkları:")for s, a, n in zip(CAMPAIGN_TYPES, agirlik, adet):    print(f"  {s:<22} n={n:>4}  ağırlık={a:.3f}")class AgirlikliTrainer(Trainer):    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):        labels = inputs.pop("labels")        out = model(**inputs)        w = torch.tensor(agirlik, dtype=torch.float32, device=out.logits.device)        kayip = nn.CrossEntropyLoss(weight=w if SINIF_AGIRLIGI else None)(            out.logits.view(-1, len(CAMPAIGN_TYPES)), labels.view(-1))        return (kayip, out) if return_outputs else kayipdef compute_metrics(pred):    """Erken durdurma ölçütü — DEPO ile aynı makro-F1."""    tahmin = np.argmax(pred.predictions, axis=-1)    s = olc([id2etiket[int(i)] for i in pred.label_ids],            [id2etiket[int(i)] for i in tahmin])    return {"accuracy": s["accuracy"], "macro_f1": s["macro_f1"]}args = TrainingArguments(    output_dir="/content/berturk_ciktilar",    learning_rate=2e-5,    per_device_train_batch_size=16,    per_device_eval_batch_size=32,    num_train_epochs=8,    weight_decay=0.01,    warmup_ratio=0.1,    eval_strategy="epoch",    save_strategy="epoch",    load_best_model_at_end=True,    metric_for_best_model="macro_f1",   # accuracy DEĞİL — dengesiz küme    greater_is_better=True,    save_total_limit=2,    logging_strategy="epoch",    seed=TOHUM,    fp16=torch.cuda.is_available(),    report_to="none",                    # harici servis yok)trainer = AgirlikliTrainer(    model=model, args=args,    train_dataset=ds["eğitim"], eval_dataset=ds["doğrulama"],    data_collator=DataCollatorWithPadding(tokenizer),    compute_metrics=compute_metrics,    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],)print("\n✅ Trainer hazır.")

In [ ]:
# --- Eğitim ---------------------------------------------------------------egitim_sonucu = trainer.train()print("\n--- eğitim özeti ---")for k, v in egitim_sonucu.metrics.items():    print(f"  {k}: {v}")print("\n--- epoch geçmişi (aşırı öğrenme burada görünür) ---")print(f"{'epoch':>6}{'eğitim kaybı':>15}{'doğrulama kaybı':>18}{'makro-F1':>11}")gecmis = {}for kayit in trainer.state.log_history:    e = kayit.get("epoch")    if e is None:        continue    g = gecmis.setdefault(round(e, 2), {})    g.update({k: v for k, v in kayit.items() if k != "epoch"})for e in sorted(gecmis):    g = gecmis[e]    tl = g.get("loss"); vl = g.get("eval_loss"); f1 = g.get("eval_macro_f1")    print(f"{e:>6.0f}{(f'{tl:.4f}' if tl is not None else '—'):>15}"          f"{(f'{vl:.4f}' if vl is not None else '—'):>18}"          f"{(f'{f1:.3f}' if f1 is not None else '—'):>11}")print("\nNot: eğitim kaybı düşerken doğrulama kaybı yükseliyorsa aşırı öğrenme")print("başlamıştır. load_best_model_at_end=True en iyi epoch'a geri döner.")

---## 5. Değerlendirme### 5a. Test kümesi (n≈76, gümüş etiketli, iç ölçüm)Bu ölçüm modelin *kendi dağılımındaki* başarısını gösterir: aynı klasik bankakorpusu, aynı gümüş etiketleme süreci. **Temel çizgiyle doğrudankarşılaştırılamaz** — temel çizgi başka bir küme üzerinde (gold, n=20)ölçüldü. Karşılaştırma 5b'de.

In [ ]:
# --- Test kümesi değerlendirmesi ------------------------------------------import numpy as np, torchfrom datasets import DatasetCEKIMSER_ESIGI = 0.0      # 0,0 = çekimserlik yok. Yükseltmek makro-F1'i DÜŞÜRÜR.def tahmin_et(dataset_veya_metinler, esik=CEKIMSER_ESIGI):    """Etiket listesi döner; güven eşiğin altındaysa None (çekimser)."""    if isinstance(dataset_veya_metinler, list):        d = Dataset.from_dict({"text": dataset_veya_metinler}).map(            tokenle, batched=True, remove_columns=["text"])    else:        d = dataset_veya_metinler    ham = trainer.predict(d).predictions    olasilik = torch.softmax(torch.tensor(ham), dim=-1).numpy()    en_iyi = olasilik.argmax(axis=-1)    guven = olasilik.max(axis=-1)    return [None if g < esik else id2etiket[int(i)]            for i, g in zip(en_iyi, guven)], guventest_gercek = [id2etiket[int(i)] for i in ds["test"]["label"]]test_tahmin, test_guven = tahmin_et(ds["test"].remove_columns(["doc_id"]))s_test = olc(test_gercek, test_tahmin)yazdir("BERTurk — test kümesi (gümüş, n=%d)" % s_test["n"], s_test)print(f"\nortalama güven: {test_guven.mean():.3f}  |  medyan: {np.median(test_guven):.3f}")

In [ ]:
# --- Karışıklık matrisi (test kümesi) -------------------------------------import numpy as npkisa = {s: s[:11] for s in CAMPAIGN_TYPES}M = np.zeros((len(CAMPAIGN_TYPES), len(CAMPAIGN_TYPES)), dtype=int)cek = np.zeros(len(CAMPAIGN_TYPES), dtype=int)for g, p in zip(test_gercek, test_tahmin):    i = CAMPAIGN_TYPES.index(g)    if p is None:        cek[i] += 1    else:        M[i, CAMPAIGN_TYPES.index(p)] += 1print("KARIŞIKLIK MATRİSİ — satır: gerçek, sütun: tahmin\n")print(f"{'':<13}" + "".join(f"{kisa[s]:>12}" for s in CAMPAIGN_TYPES) + f"{'çekims.':>9}")for i, s in enumerate(CAMPAIGN_TYPES):    print(f"{kisa[s]:<13}" + "".join(        f"{M[i, j]:>12}" if i != j else f"{('['+str(M[i, j])+']'):>12}"        for j in range(len(CAMPAIGN_TYPES))) + f"{cek[i]:>9}")print("\n[köşegen] = doğru tahminler.")

### 5b. Gold küme (n=20) — temel çizgiyle **yan yana** karşılaştırmaBu, defterin asıl sınavıdır. Kural temel çizgisi `data/gold/gold.v1.json`üzerinde ölçüldü; BERTurk **aynı küme, aynı metrik, aynı çekimser muamelesi**ile ölçülür.**Kritik bağlam — alan kayması (domain shift):** Ölçülmüş olgu, gold ile gümüşkümenin **hiç kesişmemesidir** (kesişim = 0 belge; hazırlık hücresi bunudoğruluyor). Ama kesişmemenin sebebi sadece dikkatli ayırma değil:- **Gümüş küme (eğitim):** klasik bankalar — `yapi-kredi`, `garanti-bbva`,  `qnb`, `akbank`, `ziraat-bankasi`, `is-bankasi`, `halkbank`, `vakifbank`,  `ing`, `denizbank`, `teb`.- **Gold küme (test):** katılım bankaları — `albaraka`, `vakif-katilim`,  `turkiye-finans` vb.Yani BERTurk **klasik bankacılık dilinde** eğitilip **katılım bankacılığıdilinde** sınanıyor. Katılım bankacılığında "faiz" yerine "kâr payı", "kredi"yerine "finansman" kullanılır. Bu, sızıntısızlığın en katı biçimidir ama aynızamanda BERTurk aleyhine zorlu bir sınavdır. Kural sınıflandırıcısı her ikisözcük dağarcığını da elle içerdiği için bu kaymadan etkilenmez.**Bu, sonucu yorumlarken açıkça söylenmesi gereken bir sınırdır** — BERTurkgold'da kaybederse bu "BERTurk kötü" demek değil, "eğitim kümesi hedef alanıtemsil etmiyor" demek olabilir.

In [ ]:
# --- Gold kümesi: temel çizgiyle YAN YANA ---------------------------------# ÖLÇÜLMÜŞ temel çizgi — `python -m scripts.eval_classifier` çıktısı# (data/gold/gold.v1.json, n=20). Uydurulmuş değil, kopyalanmıştır.TEMEL_CIZGI = {"ad": "kural (RuleHintClassifier)",               "accuracy": 0.700, "macro_f1": 0.762, "cekimser": 1, "n": 20}gold_gercek = [r["label"] for r in gold_kayitlar]gold_tahmin, gold_guven = tahmin_et([r["text"] for r in gold_kayitlar])s_gold = olc(gold_gercek, gold_tahmin)yazdir("BERTurk — gold kümesi (n=%d)" % s_gold["n"], s_gold)# --- KARŞILAŞTIRMA TABLOSU ---print("\n" + "=" * 62)print("KARŞILAŞTIRMA — aynı gold küme (n=20), aynı metrik")print("=" * 62)print(f"{'kol':<28}{'accuracy':>11}{'makro-F1':>11}{'çekimser':>11}")print("-" * 62)print(f"{TEMEL_CIZGI['ad']:<28}{TEMEL_CIZGI['accuracy']:>11.3f}"      f"{TEMEL_CIZGI['macro_f1']:>11.3f}{TEMEL_CIZGI['cekimser']:>11}")print(f"{'BERTurk (ince ayarlı)':<28}{s_gold['accuracy']:>11.3f}"      f"{s_gold['macro_f1']:>11.3f}{s_gold['cekimser']:>11}")print("-" * 62)d_acc = s_gold["accuracy"] - TEMEL_CIZGI["accuracy"]d_f1  = s_gold["macro_f1"] - TEMEL_CIZGI["macro_f1"]print(f"{'FARK (BERTurk - kural)':<28}{d_acc:>+11.3f}{d_f1:>+11.3f}")print("=" * 62)if abs(d_f1) < 0.05:    print("\n⚠️  Makro-F1 farkı n=20'de gürültüden ayırt EDİLEMEZ.")    print("    KAZANAN İLAN ETME. Bkz. bölüm 6.")elif d_f1 > 0:    print(f"\nBERTurk makro-F1'de {d_f1:+.3f} önde — ama n=20'de bu tek")    print("belgenin yer değiştirmesiyle oynayabilir. Bölüm 6'daki güven")    print("aralığını okumadan karar verme.")else:    print(f"\nBERTurk makro-F1'de {d_f1:+.3f} GERİDE. Kural çizgisi korunur.")    print("Olası sebep: alan kayması (klasik banka → katılım bankası).")

In [ ]:
# --- Tahminleri dışa aktar: scripts/eval_classifier.py ile uyumlu ---------import json# Şema: {"doc_id": ..., "label": ...}  — label: null = çekimserwith open("/content/berturk_preds.jsonl", "w", encoding="utf-8") as fh:    for r, p in zip(gold_kayitlar, gold_tahmin):        fh.write(json.dumps({"doc_id": r["doc_id"], "label": p},                            ensure_ascii=False) + "\n")print("-> /content/berturk_preds.jsonl")print("\nDepoya taşıdıktan sonra (data/eval/berturk_preds.jsonl):")print("  python -m scripts.eval_classifier \\")print("      --predictions data/eval/berturk_preds.jsonl \\")print("      --name berturk --compare")print("\nBu komut yukarıdaki tabloyu DEPODA yeniden üretir. İki sayı")print("tutmuyorsa ölçüm hattında bir fark var demektir — araştır.")

---## 6. İstatistiksel uyarı — n=505 ve 8 sınıf**Bu defterin ürettiği hiçbir sayı tek başına "kazanan" ilan etmeye yetmez.**| Sorun | Somut durum ||---|---|| **Gold küme çok küçük** | n=**20**, 8 sınıf → sınıf başına **2,5** örnek. Tek belgenin doğru/yanlış gitmesi accuracy'yi 0,05 oynatır; makro-F1'i daha fazla. || **Eğitim kümesi küçük** | n=**505** → test kümesi ≈76, sınıf başına ≈9,5 örnek. || **Tek bölme** | Tek bir `random_state=42` bölmesinin sonucu raporlanıyor. Farklı tohum farklı sayı verir; bu değişkenlik ölçülmedi. || **Güven aralığı yok** | Nokta tahmini karşılaştırmak, aralıklar örtüşüyorsa yanıltıcıdır. || **Gümüş etiket gürültülü** | Etiketler insan değil, LLM uzlaşmasıyla üretildi (255 kayıt üç-oy-aynı, 250 kayıt iki-LLM-aynı). Tavan, etiketleyicinin doğruluğudur. || **Alan kayması** | Eğitim klasik banka, gold katılım bankası (bölüm 5b). |Aşağıdaki hücre en azından **güven aralığını** ölçer (bootstrap). Tek bölme veetiket gürültüsü sorunları ölçülmedi — bunlar açık kalan risklerdir.> **Kabul kuralı:** BERTurk projeye ancak gold makro-F1'inin **%95 güven> aralığının alt sınırı** kural çizgisinin **0,762** değerini aşarsa alınır.> Aralık 0,762'yi içeriyorsa sonuç "ayırt edilemez"dir ve varsayılan> `RuleHintClassifier` olarak kalır (basit olan kazanır).

In [ ]:
# --- Bootstrap güven aralığı ----------------------------------------------import numpy as npdef bootstrap_ca(gercekler, tahminler, yineleme=5000, tohum=TOHUM):    rng = np.random.default_rng(tohum)    n = len(gercekler)    acc, f1 = [], []    for _ in range(yineleme):        idx = rng.integers(0, n, size=n)          # yerine koyarak örnekleme        s = olc([gercekler[i] for i in idx], [tahminler[i] for i in idx])        acc.append(s["accuracy"]); f1.append(s["macro_f1"])    return np.array(acc), np.array(f1)for ad, gercek, tahmin in (("GOLD (n=%d)" % len(gold_gercek), gold_gercek, gold_tahmin),                           ("TEST (n=%d)" % len(test_gercek), test_gercek, test_tahmin)):    acc, f1 = bootstrap_ca(gercek, tahmin)    a_lo, a_hi = np.percentile(acc, [2.5, 97.5])    f_lo, f_hi = np.percentile(f1,  [2.5, 97.5])    print(f"\n--- {ad} — %95 bootstrap güven aralığı (5000 yineleme) ---")    print(f"  accuracy : {acc.mean():.3f}  [{a_lo:.3f}, {a_hi:.3f}]  genişlik={a_hi-a_lo:.3f}")    print(f"  makro-F1 : {f1.mean():.3f}  [{f_lo:.3f}, {f_hi:.3f}]  genişlik={f_hi-f_lo:.3f}")    if ad.startswith("GOLD"):        print(f"\n  KABUL KRİTERİ: alt sınır ({f_lo:.3f}) > temel çizgi (0,762)?")        if f_lo > 0.762:            print("  ✅ GEÇTİ — BERTurk ölçülebilir biçimde üstün. Projeye alınabilir.")        elif f_hi < 0.762:            print("  ❌ KALDI — BERTurk ölçülebilir biçimde GERİDE. Kural çizgisi korunur.")        else:            print("  ⚠️  AYIRT EDİLEMEZ — aralık 0,762'yi İÇERİYOR.")            print("     KAZANAN İLAN ETME. Varsayılan RuleHintClassifier kalır.")            print("     Gerekli: daha büyük gold küme veya çok-tohumlu tekrar.")print("\n" + "!" * 62)print("HATIRLATMA: Bu aralıklar TEK bölmenin (random_state=42) içindeki")print("örnekleme belirsizliğini ölçer. BÖLME değişkenliğini ölçmez.")print("Yayına giden bir iddia için 5 farklı tohumla tekrar koşulmalı.")print("!" * 62)

---## 7. Modeli dışa aktar ve projeye geri taşı

In [ ]:
# --- Model dışa aktarma ---------------------------------------------------import os, shutil, jsonKAYIT_DIR = "/content/berturk-kampanya-8sinif"trainer.save_model(KAYIT_DIR)          # config.json + model.safetensorstokenizer.save_pretrained(KAYIT_DIR)   # vocab.txt + tokenizer_config.json# Kaynak/lisans künyesi — ağırlıklar depodan bağımsız dolaşırsa da izlenebilsinwith open(os.path.join(KAYIT_DIR, "KUNYE.json"), "w", encoding="utf-8") as fh:    json.dump({        "taban_model": MODEL_ADI,        "taban_lisans": "MIT",        "taban_lisans_kanit": "https://huggingface.co/dbmdz/bert-base-turkish-cased",        "base_model_zinciri": "yok (kök model)",        "gorev": "8 sınıflı kampanya türü sınıflandırması",        "siniflar": CAMPAIGN_TYPES,        "egitim_kumesi": "data/silver/silver.jsonl (n=505, gümüş etiket)",        "bolme": {"eğitim": len(idx_egitim), "doğrulama": len(idx_dogrulama),                  "test": len(idx_test), "random_state": TOHUM},        "max_length": MAX_LENGTH,        "gold_accuracy": round(s_gold["accuracy"], 4),        "gold_macro_f1": round(s_gold["macro_f1"], 4),        "temel_cizgi_gold": {"accuracy": 0.700, "macro_f1": 0.762},    }, fh, ensure_ascii=False, indent=2)print("kaydedilen dosyalar:")for f in sorted(os.listdir(KAYIT_DIR)):    print(f"  {f:<28}{os.path.getsize(os.path.join(KAYIT_DIR, f))/1e6:>8.1f} MB")arsiv = shutil.make_archive("/content/berturk-kampanya-8sinif", "zip",                            root_dir="/content", base_dir="berturk-kampanya-8sinif")print(f"\narşiv: {arsiv}  ({os.path.getsize(arsiv)/1e6:.1f} MB)")# Drive'a kopyala (indirme ücretsiz katmanda kopabilir)if os.path.isdir("/content/drive/MyDrive"):    os.makedirs(DRIVE_KLASOR, exist_ok=True)    shutil.copy(arsiv, DRIVE_KLASOR)    shutil.copy("/content/berturk_preds.jsonl", DRIVE_KLASOR)    print(f"-> Drive: {DRIVE_KLASOR}/")else:    from google.colab import files    files.download(arsiv)

### Projeye geri taşıma — hangi klasör, hangi dosya, `src/` içinde nereye bağlanır**1. Arşivi aç.** Depo kökünde (`app/`):```bashmkdir -p modelsunzip ~/Downloads/berturk-kampanya-8sinif.zip -d models/# sonuç: models/berturk-kampanya-8sinif/{config.json,model.safetensors,vocab.txt,...}````models/` **`.gitignore`'da** (satır 81) — ağırlıklar depoya commit edilmez,bu kasıtlıdır. Ağırlıklar sürüm dışı bir kanalla (Drive/release) taşınır.**2. Tahminleri depoya koy ve ölçümü depoda tekrarla:**```bashcp ~/Downloads/berturk_preds.jsonl data/eval/python -m scripts.eval_classifier \    --predictions data/eval/berturk_preds.jsonl --name berturk --compare```Bu komut bölüm 5b'deki tabloyu depoda yeniden üretmeli. **Sayılar tutmuyorsasonucu raporlama** — ölçüm hattında fark var demektir.**3. `src/` içinde bağlantı noktası — kod değişikliği GEREKMİYOR.**Bağlantı noktası zaten hazır: `src/extraction/ner/classifier.py`| Bileşen | Rol ||---|---|| `RuleHintClassifier` | Kural temel çizgisi; her zaman çalışır, sıfır bağımlılık. || `BerturkClassifier(model_dir=...)` | İnce ayarlı yol. `model_dir` yoksa **sessizce** `RuleHintClassifier`'a düşer. || `default_classifier()` | Ortama bakar: ağırlık varsa BERTurk, yoksa kural. |`BerturkClassifier.__init__` model dizinini **`BERTURK_MODEL_DIR`** ortamdeğişkeninden okur. Etkinleştirmek için (bkz. `.env.example` satır 45):```bashexport BERTURK_MODEL_DIR=$(pwd)/models/berturk-kampanya-8sinif```Docker'da `docker-compose.yml` zaten `./models` dizinini konteynere bağlıyor;oradaki yol `/models/berturk-kampanya-8sinif` olur.**4. Etiket uyumu şart.** `BerturkClassifier.classify`, model etiketi`CAMPAIGN_TYPES` içinde değilse kurala düşer. Bu defter `id2label`'ı doğrudan`CAMPAIGN_TYPES`'tan kurduğu için uyum sağlanmıştır — model dizinindeki`config.json` elle düzenlenirse bozulur.**5. Devreye alma kararı bölüm 6'nın kabul kriterine bağlıdır.** Kritergeçilmediyse `BERTURK_MODEL_DIR` **ayarlanmaz** ve sistem kural çizgisindekalır.

---## 8. ÇEVRİMDIŞI NOTU — çıkarım internetsiz koşmak zorundaYarışma sistemi **çevrimdışı** çalışmak zorundadır (`CLAUDE.md` §3;`docs/OFFLINE-KANIT.md`). Bu deftere özgü ayrım:| Aşama | Ortam | Ağ ||---|---|---|| **Eğitim** | Google Colab | çevrimiçi (ağırlık indirilir) — **kabul edilebilir**, eğitim teslim edilen sistemin parçası değildir || **Çıkarım** | Yerel / Docker | **çevrimdışı zorunlu** — ağırlıklar diskten okunur |İnce ayarlı model dizini `config.json`, `model.safetensors`, `vocab.txt` ve`tokenizer_config.json` dosyalarının hepsini içerir; yani `transformers`Hub'a hiç gitmeden yükleyebilir. Kritik nokta **`local_files_only=True`** ve`HF_HUB_OFFLINE=1` — bunlar olmadan `transformers` eksik bir dosya içinsessizce ağa çıkar ve çevrimdışı makinede çalışma zamanında patlar.Aşağıdaki hücre bunu **kanıtlar**: ağ değişkenlerini kapatır, modeli yalnızcayerel dosyalardan yükler ve bir tahmin üretir. Bu hücre Colab'da da koşar(model zaten diskte olduğu için), ama asıl yeri yerel makinedir.

In [ ]:
# --- ÇEVRİMDIŞI ÇIKARIM DOĞRULAMASI ---------------------------------------# Bu hücre YEREL makinede de aynen koşar. Colab'da koşarken bile ağa çıkmaz.import osos.environ["HF_HUB_OFFLINE"] = "1"        # docker-compose.yml ile aynıos.environ["TRANSFORMERS_OFFLINE"] = "1"from transformers import (AutoTokenizer, AutoModelForSequenceClassification,                          pipeline)MODEL_DIR = os.environ.get("BERTURK_MODEL_DIR", "/content/berturk-kampanya-8sinif")# local_files_only=True: dosya eksikse AĞA ÇIKMAZ, HATA VERİR. İstenen budur —# sessizce indirmek yerine, çevrimdışı ortamda eksik dosyayı erkenden bildirir.tok_off = AutoTokenizer.from_pretrained(MODEL_DIR, local_files_only=True)mdl_off = AutoModelForSequenceClassification.from_pretrained(    MODEL_DIR, local_files_only=True)boru = pipeline("text-classification", model=mdl_off, tokenizer=tok_off,                top_k=1, device=-1)       # device=-1 → CPU (demo donanımı)ornekler = [    "Konut finansmanında 120 aya varan vade ve %2,89 kâr payı oranı fırsatı.",    "Yeni müşterilerimize ilk alışverişte 500 TL bonus hediye!",    "Sıfır ve ikinci el taşıt finansmanında 36 ay vade avantajı.",]print(f"model dizini: {MODEL_DIR}")print(f"ağ değişkenleri: HF_HUB_OFFLINE={os.environ['HF_HUB_OFFLINE']} "      f"TRANSFORMERS_OFFLINE={os.environ['TRANSFORMERS_OFFLINE']}\n")for m in ornekler:    r = boru(m)    top = r[0][0] if isinstance(r[0], list) else r[0]    print(f"  {top['label']:<22} ({top['score']:.3f})  ← {m[:52]}...")print("\n✅ Çevrimdışı çıkarım doğrulandı: ağ kapalıyken yerel ağırlıklarla")print("   tahmin üretildi. src/extraction/ner/classifier.py:BerturkClassifier")print("   aynı dizini BERTURK_MODEL_DIR üzerinden okur.")

---## 9. Kapanış — sonucu nasıl okumalıBu defterin ürettiği sayı, ancak aşağıdaki üç koşulun hepsi sağlanırsaprojeye alınacak bir iddiaya dönüşür:1. **Kabul kriteri geçildi** — gold makro-F1'inin %95 güven aralığının alt   sınırı 0,762'yi aşıyor (bölüm 6).2. **Ölçüm depoda tekrarlandı** — `python -m scripts.eval_classifier   --predictions data/eval/berturk_preds.jsonl --name berturk --compare`   defterdeki tabloyu birebir yeniden üretti.3. **Çevrimdışı çıkarım doğrulandı** — bölüm 8 hücresi ağ kapalıyken geçti.Üçü de sağlanırsa sonuç `docs/rapor/ablasyon.md` tablosuna satır olarakgirer ve `BERTURK_MODEL_DIR` devreye alınır.**Sağlanmazsa** — özellikle güven aralığı 0,762'yi içeriyorsa — doğru karar`RuleHintClassifier`'da kalmaktır. Ayırt edilemez bir farkı "iyileştirme"diye raporlamak, bu projenin `CLAUDE.md` §19'daki "uydurma yok" kuralınınölçüm tarafındaki ihlalidir.### Açık kalan riskler (ölçülmedi)- **Tek bölme.** Yalnız `random_state=42` koşuldu; bölme değişkenliği bilinmiyor.- **Gümüş etiket gürültüsü.** Etiketler LLM uzlaşmasıyla üretildi; insan  doğrulaması yapılmadı. Modelin tavanı etiketleyicinin doğruluğudur.- **Alan kayması.** Eğitim klasik banka, değerlendirme katılım bankası.  Katılım bankası verisiyle eğitim yapılmadı.- **Gold küme n=20.** Sınıf başına 2,5 örnek; en zayıf halka budur.Planın tamamı: `docs/rapor/berturk-ince-ayar-plani.md`